In [143]:
print("hello")

hello


In [144]:
import pandas as pd

In [145]:
data=pd.read_csv("twitter_training.csv")

In [146]:
data.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [147]:
data=data.sample(15000,random_state=42)

In [148]:
data.shape

(15000, 4)

In [149]:
## Data Exploration 

In [150]:
data['Positive'].value_counts()

Positive
Negative      4488
Positive      4275
Neutral       3568
Irrelevant    2669
Name: count, dtype: int64

In [151]:
data.isnull().sum()

2401                                                       0
Borderlands                                                0
Positive                                                   0
im getting on borderlands and i will murder you all ,    151
dtype: int64

In [152]:
data.duplicated().sum()

np.int64(127)

In [153]:
data=data.dropna()

In [154]:
data.isnull().sum()

2401                                                     0
Borderlands                                              0
Positive                                                 0
im getting on borderlands and i will murder you all ,    0
dtype: int64

In [155]:
data.drop_duplicates(inplace=True)

In [156]:
data.duplicated().sum()

np.int64(0)

In [157]:
data.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
34877,6792,Fortnite,Irrelevant,went to go in george's room to find his door w...
21704,4115,CS-GO,Positive,Yo this looks LIT! Team:GO/Overwatch combo
47008,5665,HomeDepot,Negative,Pay attention executive administrators. While ...
7969,9369,Overwatch,Irrelevant,Guy looked at me and says my name was put on t...
454,2476,Borderlands,Positive,one


In [158]:
data.drop(columns=['2401','Borderlands'],inplace=True)

In [159]:
data.head(1)

,Positive,"im getting on borderlands and i will murder you all ,"
34877,Irrelevant,went to go in george's room to find his door w...


In [160]:
data.rename(columns={
    'Positive':'Sentiment',
    'im getting on borderlands and i will murder you all ,':'Tweets'
    },inplace=True)

In [161]:
data.head()

,Sentiment,Tweets
34877,Irrelevant,went to go in george's room to find his door w...
21704,Positive,Yo this looks LIT! Team:GO/Overwatch combo
47008,Negative,Pay attention executive administrators. While ...
7969,Irrelevant,Guy looked at me and says my name was put on t...
454,Positive,one


In [162]:
data['Sentiment'].value_counts()

Sentiment
Negative      4418
Positive      4201
Neutral       3508
Irrelevant    2619
Name: count, dtype: int64

In [163]:
data=data[(data['Sentiment']=='Positive') | (data['Sentiment']=='Negative') |
  (data['Sentiment']=='Neutral') ]

In [164]:
data.head()

,Sentiment,Tweets
21704,Positive,Yo this looks LIT! Team:GO/Overwatch combo
47008,Negative,Pay attention executive administrators. While ...
454,Positive,one
58076,Negative,@Rainbow6Game Server are Available in Xbox 🥺
40659,Positive,so Awesome siege attempt... strangest bleeding...


In [165]:
data['Sentiment'].value_counts()

Sentiment
Negative    4418
Positive    4201
Neutral     3508
Name: count, dtype: int64

In [166]:
# Feature Encoding 
from sklearn.preprocessing import LabelEncoder

In [167]:
label = LabelEncoder()

In [168]:
data['Sentiment']=label.fit_transform(data['Sentiment'])

In [169]:
data['Sentiment']

21704    2
47008    0
454      2
58076    0
40659    2
        ..
57537    2
7757     2
3887     0
22629    1
56434    2
Name: Sentiment, Length: 12127, dtype: int64

In [170]:
from tensorflow.keras.preprocessing.text import Tokenizer


# Create a NEW tokenizer
tokenizer = Tokenizer()

# Fit tokenizer
tokenizer.fit_on_texts(data['Tweets']) # fit = learn the vocabulary

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(data['Tweets'])

In [171]:
from keras.utils import pad_sequences

In [172]:
# A neural network generally needs inputs with the same shape/length.
sequences=pad_sequences(sequences,padding='pre')

"I love this"             → [5, 1, 3]

"I love this movie"       → [5, 1, 3, 2]

"I really love this movie" → [5, 6, 1, 3, 2]

sequences=pad_sequences(sequences,padding='pre')

[0, 0, 5, 1, 3]
[0, 5, 1, 3, 2]
[5, 6, 1, 3, 2]


In [173]:
sequences

array([[    0,     0,     0, ...,    94,   196,  2524],
       [    0,     0,     0, ...,     6,  2162,  3097],
       [    0,     0,     0, ...,     0,     0,    51],
       ...,
       [    0,     0,     0, ...,    19,   222,  6167],
       [    0,     0,     0, ...,    42,    73, 17871],
       [    0,     0,     0, ...,   791,  1030,  6491]],
      shape=(12127, 99), dtype=int32)

In [174]:
from tensorflow.keras.layers import SimpleRNN,Dense,Input,Embedding
from tensorflow.keras.models import Sequential

In [175]:
X=sequences

In [176]:
y=data['Sentiment'].values

In [177]:
y

array([2, 0, 2, ..., 0, 1, 2], shape=(12127,))

In [178]:
from sklearn.model_selection import train_test_split

In [179]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [180]:
X_train.shape

(9701, 99)

In [181]:
model=Sequential()
model.add(SimpleRNN(32,input_shape=(99,1),return_sequences=False))
model.add(Dense(3,activation='softmax'))

d:\aashish\generative ai course\Deep Learning\RNN\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [182]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,187 (4.64 KB)

 Trainable params: 1,187 (4.64 KB)

 Non-trainable params: 0 (0.00 B)

In [183]:
model.compile(optimizer="adam",metrics=['accuracy'],loss='sparse_categorical_crossentropy')

In [184]:
model.fit(X_train,y_train,epochs=10,validation_data=(X_test,y_test),batch_size=64)

Epoch 1/10


152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.3570 - loss: 1.1327 - val_accuracy: 0.3702 - val_loss: 1.0922
Epoch 2/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.3746 - loss: 1.0934 - val_accuracy: 0.3454 - val_loss: 1.1054
Epoch 3/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.3578 - loss: 1.0934 - val_accuracy: 0.3607 - val_loss: 1.0952
Epoch 4/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.3674 - loss: 1.0940 - val_accuracy: 0.3660 - val_loss: 1.0944
Epoch 5/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.3741 - loss: 1.0893 - val_accuracy: 0.3664 - val_loss: 1.0926
Epoch 6/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.3720 - loss: 1.0886 - val_accuracy: 0.3491 - val_loss: 1.0949
Epoch 7/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.3708 - loss: 1.0880 - val_accuracy: 0.3718 - val_loss: 1.0906
Epoch 8/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.3697 - loss: 1.0867 - val_accuracy: 0.370

In [185]:
vocab_size=len(tokenizer.word_index)+1

In [186]:
vocab_size

17872

In [187]:
model_2=Sequential()
model_2.add(Input(shape=(99,)))
model_2.add(Embedding(input_dim=vocab_size,output_dim=64))
model_2.add(Dense(3,activation='softmax'))

In [188]:
model_2.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 99, 64)         │     1,143,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 99, 3)          │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,144,003 (4.36 MB)

 Trainable params: 1,144,003 (4.36 MB)

 Non-trainable params: 0 (0.00 B)

In [189]:
model_final=Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=99,
        mask_zero=True
        ),
SimpleRNN(32),
Dense(3,activation='softmax')
])

d:\aashish\generative ai course\Deep Learning\RNN\venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [190]:
model_final.compile(
    optimizer='adam',
    metrics=['accuracy'],
    loss='sparse_categorical_crossentropy',
    )

In [191]:
model_final.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [197]:
model_final.fit(X_train,y_train,epochs=100,validation_data=(X_test,y_test),batch_size=64)

Epoch 1/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 18s 119ms/step - accuracy: 0.9815 - loss: 0.0374 - val_accuracy: 0.6987 - val_loss: 1.1538
Epoch 2/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 21s 118ms/step - accuracy: 0.9818 - loss: 0.0358 - val_accuracy: 0.6917 - val_loss: 1.1891
Epoch 3/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 21s 120ms/step - accuracy: 0.9809 - loss: 0.0360 - val_accuracy: 0.6908 - val_loss: 1.2191
Epoch 4/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 23s 150ms/step - accuracy: 0.9825 - loss: 0.0350 - val_accuracy: 0.6921 - val_loss: 1.2569
Epoch 5/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 35s 109ms/step - accuracy: 0.9821 - loss: 0.0359 - val_accuracy: 0.6789 - val_loss: 1.2683
Epoch 6/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 19s 123ms/step - accuracy: 0.9644 - loss: 0.0905 - val_accuracy: 0.6641 - val_loss: 1.2190
Epoch 7/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 18s 119ms/step - accuracy: 0.9663 - loss: 0.0797 - val_accuracy: 0.6653 - val_loss: 1.2623
Epoch 8/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 19s 122ms/step - accuracy: 0.9802 -

In [193]:
import pickle 

In [195]:
with open ('tokenizer.pkl','wb') as file:
    pickle.dump(tokenizer,file)

In [198]:
model_final.save('model.h5')